In [75]:
import torch
from torch import nn
import torch.nn.functional as F


In [76]:
device = "mps" if torch.backends.mps.is_available() else "cpu" # mac

In [77]:
import pandas as pd

In [39]:
df = pd.read_csv("../data/Recipe_Clean.csv")

text = "".join(df["text"])

characters = sorted(list(set(text)))
vocab_size = len(characters)

char_to_idx = { ch:i for i,ch in enumerate(characters) }
idx_to_char = { i:ch for i,ch in enumerate(characters) }
encode = lambda xs: [char_to_idx[x] for x in xs] # encoder: take the string, output the list of integers
decode = lambda xs: ''.join([idx_to_char[x] for x in xs]) # decoder: take the list of integers, output the string


In [78]:
#speical tokens

df = pd.read_csv("../data/Recipe_Clean_Special_Tokens.csv")

text = "".join(df["text"])

characters = sorted(list(set(text)))
vocab_size = len(characters)

char_to_idx = { ch:i for i,ch in enumerate(characters) }
idx_to_char = { i:ch for i,ch in enumerate(characters) }
encode = lambda xs: [char_to_idx[x] for x in xs] # encoder: take the string, output the list of integers
decode = lambda xs: ''.join([idx_to_char[x] for x in xs]) # decoder: take the list of integers, output the string


In [79]:
%pip install tiktoken
import tiktoken

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [90]:
#special tokens
base_encoding = tiktoken.get_encoding("gpt2")

SPECIAL_TOKENS = {
    "<|recipe_start|>": 50257,
    "<|ingredients|>": 50258,
    "<|title|>": 50259,
    "<|instructions|>": 50260,
    "<|recipe_end|>": 50261,
}

enc = tiktoken.Encoding(
    name="gpt2_recipe",
    pat_str=base_encoding._pat_str,
    mergeable_ranks=base_encoding._mergeable_ranks,
    special_tokens={
        **base_encoding._special_tokens,
        **SPECIAL_TOKENS,
    },
)


ALLOWED_SPECIAL = set(SPECIAL_TOKENS.keys())

def encode(text):
    return enc.encode(
        text,
        allowed_special=ALLOWED_SPECIAL
    )

def decode(token_ids):
    return enc.decode(token_ids)

# The custom tokenizer now has 50,262 token IDs:
# 50,257 original GPT-2 IDs + 5 new special tokens
vocab_size = enc.n_vocab

print("Tokenizer: Custom GPT-2 recipe tokenizer")
print("Vocabulary size:", vocab_size)
print("Registered special tokens:", enc._special_tokens)

Tokenizer: Custom GPT-2 recipe tokenizer
Vocabulary size: 50262
Registered special tokens: {'<|endoftext|>': 50256, '<|recipe_start|>': 50257, '<|ingredients|>': 50258, '<|title|>': 50259, '<|instructions|>': 50260, '<|recipe_end|>': 50261}


In [ ]:
#normal tokenizer
# Load the same GPT-2 tokenizer used in your tokenization notebook
enc = tiktoken.get_encoding("gpt2")

# The GPT-2 tokenizer contains 50,257 possible token IDs
vocab_size = enc.n_vocab

def encode(text):
    """Convert a string into a list of tiktoken token IDs."""
    return enc.encode(text)

def decode(token_ids):
    """Convert a list of token IDs back into text."""
    return enc.decode(token_ids)

print("Tokenizer: GPT-2 tiktoken")
print("Vocabulary size:", vocab_size)

sample = text[:200]
sample_tokens = encode(sample)

print("Sample text:")
print(sample)

print("\nToken IDs:")
print(sample_tokens[:30])

print("\nDecoded text:")
print(decode(sample_tokens))

/Users/alinakenny/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Tokenizer: GPT-2 tiktoken
Vocabulary size: 50257
Sample text:
Title: Air Fryer Potato Slices with Dipping Sauce

Ingredients: 3/4 cup ketchup
1/2 cup beer
1 tablespoon Worcestershire sauce
1/2 teaspoon onion powder
1/4 teaspoon cayenne
2 baking potatoes
olive oi

Token IDs:
[19160, 25, 3701, 25305, 263, 43876, 311, 677, 274, 351, 360, 4501, 37618, 198, 198, 41222, 25, 513, 14, 19, 6508, 479, 47132, 198, 16, 14, 17, 6508, 6099, 198]

Decoded text:
Title: Air Fryer Potato Slices with Dipping Sauce

Ingredients: 3/4 cup ketchup
1/2 cup beer
1 tablespoon Worcestershire sauce
1/2 teaspoon onion powder
1/4 teaspoon cayenne
2 baking potatoes
olive oi


In [88]:
#speical token
test_text = (
    "<|recipe_start|>\n"
    "<|ingredients|>\n"
    "chicken\nrice\ncarrots\n"
    "<|title|>\n"
)

test_ids = encode(test_text)

print(test_ids)
print(decode(test_ids))



[27, 91, 29102, 431, 62, 9688, 91, 29, 198, 27, 91, 278, 23320, 91, 29, 198, 354, 5973, 198, 20970, 198, 7718, 24744, 198, 27, 91, 7839, 91, 29, 198]
<|recipe_start|>
<|ingredients|>
chicken
rice
carrots
<|title|>



In [92]:
for token, expected_id in SPECIAL_TOKENS.items():
    token_ids = encode(token)

    print(
        token,
        "->",
        token_ids,
        "expected:",
        expected_id
    )

    assert token_ids == [expected_id]


<|recipe_start|> -> [50257] expected: 50257
<|ingredients|> -> [50258] expected: 50258
<|title|> -> [50259] expected: 50259
<|instructions|> -> [50260] expected: 50260
<|recipe_end|> -> [50261] expected: 50261


*************SPEICAL TOKENS BELOW!!!

In [93]:
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import tiktoken

device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: mps


In [94]:
df = pd.read_csv(
    "../data/Recipe_Clean_Special_Tokens.csv"
)

df["text"] = df["text"].fillna("")

In [95]:
text = "\n".join(df["text"].astype(str))

In [96]:
token_ids = encode(text)

data = torch.tensor(
    token_ids,
    dtype=torch.long
)

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Recipes:", len(df))
print("Characters:", len(text))
print("Tokens:", len(data))
print("Vocabulary size:", vocab_size)
print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Recipes: 62126
Characters: 66588644
Tokens: 15911447
Vocabulary size: 50262
Training tokens: 14320302
Validation tokens: 1591145


In [97]:
generator = torch.Generator().manual_seed(42)

indices = torch.randperm(
    len(df),
    generator=generator
)

split_index = int(0.9 * len(df))

train_indices = indices[:split_index]
val_indices = indices[split_index:]

train_text = "\n".join(
    df.iloc[train_indices]["text"].tolist()
)

val_text = "\n".join(
    df.iloc[val_indices]["text"].tolist()
)

train_data = torch.tensor(
    encode(train_text),
    dtype=torch.long
)

val_data = torch.tensor(
    encode(val_text),
    dtype=torch.long
)

print("Training recipes:", len(train_indices))
print("Validation recipes:", len(val_indices))

Training recipes: 55913
Validation recipes: 6213


In [98]:
def get_batch(split, batch_size, context_size):
    source = train_data if split == "train" else val_data

    if len(source) <= context_size:
        raise ValueError(
            f"{split} data has only {len(source)} tokens, "
            f"but context_size is {context_size}."
        )

    ix = torch.randint(
        0,
        len(source) - context_size,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + context_size]
        for i in ix
    ])

    y = torch.stack([
        source[i + 1:i + context_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

***********END OF SPEICLA TOKEN

In [46]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(len(text) * 0.9)

In [47]:
token_ids = encode(text)
data = torch.tensor(token_ids, dtype=torch.long)

# Split according to the number of tokens, not the number of characters
n = int(len(data) * 0.9)

train_data = data[:n]
val_data = data[n:]

print("Number of characters:", len(text))
print("Number of tiktoken tokens:", len(data))
print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))
print("Compression ratio:", len(text) / len(data))

Number of characters: 64103604
Number of tiktoken tokens: 15878051
Training tokens: 14290245
Validation tokens: 1587806
Compression ratio: 4.037246384962487


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(len(text) * 0.9)
train_data = data[:n]
val_data = data[n:]



In [99]:
def get_batch(split, batch_size, context_size):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_size, (batch_size,))
    x = torch.stack([data[i:i+context_size] for i in ix])
    y = torch.stack([data[i+1:i+context_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [49]:
class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # lookup table, vocab_size x vocab_size

  def forward(self, idx, targets=None):
    # idx (batch_size, context_size)
    logits = self.token_embedding_table(idx) # (batch_size, context_size, vocab_size)

    if targets is not None:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    else:
      loss = None

    return logits, loss

In [ ]:
#normal
def generate(model, context_size, start_idx, number_of_tokens):
  idx = start_idx
  for _ in range(number_of_tokens):
    # crop to last block_size of tokens
    idx_cond = idx[:, -context_size:]
    logits, loss = model(idx_cond)
    # apply softmas to get probabilities
    logits = logits[:, -1, :] # (batch_size, context_size)
    probs = F.softmax(logits, dim=1) # (batch_size, context_size)
    idx_next = torch.multinomial(probs, num_samples=1) # (batch_size, 1)
    idx = torch.cat((idx, idx_next), dim=1) # (batch_size, t + 1)
  return idx

In [100]:
#speicla token

@torch.no_grad()
def generate(
    model,
    context_size,
    start_idx,
    number_of_tokens,
    end_token_id=None
):
    model.eval()

    idx = start_idx

    for _ in range(number_of_tokens):
        # Use only the most recent context_size tokens
        idx_cond = idx[:, -context_size:]

        # Get the model's predictions
        logits, loss = model(idx_cond)

        # Keep predictions for only the final position
        logits = logits[:, -1, :]

        # Convert logits into probabilities
        probs = F.softmax(
            logits,
            dim=-1
        )

        # Sample the next token
        idx_next = torch.multinomial(
            probs,
            num_samples=1
        )

        # Append it to the generated sequence
        idx = torch.cat(
            (idx, idx_next),
            dim=1
        )

        # Stop when recipe-end token is generated
        if (
            end_token_id is not None
            and idx_next.item() == end_token_id
        ):
            break

    model.train()

    return idx




In [101]:
end_token_id = SPECIAL_TOKENS[
    "<|recipe_end|>"
]

In [ ]:
generated_ids = generate(
    model=model,
    context_size=context_size,
    start_idx=start_idx,
    number_of_tokens=300,
    end_token_id=end_token_id
)

In [102]:
@torch.no_grad()
def estimate_loss(model, batch_size, context_size, eval_iters=100):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size, context_size)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


def train(model, steps, batch_size, context_size, report_frequency=1000):
  optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

  for step in range(steps): # increase number of steps for good results...
      # sample a batch of data
      xb, yb = get_batch('train', batch_size, context_size)

      # evaluate the loss
      logits, loss = model(xb, yb)
      optimizer.zero_grad(set_to_none=True)
      loss.backward()
      optimizer.step()
      if step % report_frequency == 0 or step == steps - 1:
          losses = estimate_loss(model, batch_size, context_size)
          print(f"Step {step}, train loss: {losses['train']:.4f} val loss: {losses['val']:.4f}")

In [ ]:
#for character level
def train_generate_print(model):
  train(model, 5000, 32, 8)

  start_idx = torch.zeros((1, 1),  dtype=torch.long, device=device)
  max_tokens = 300
  print(decode(
      generate(model, 8, start_idx=start_idx, number_of_tokens=max_tokens)[0].tolist()
    )
  )

In [52]:
#for tiktoken
def train_generate_print(
    model,
    steps=5000,
    batch_size=32,
    context_size=128,
    max_new_tokens=200,
    prompt="<RECIPE>\nTitle:"
):
    train(
        model=model,
        steps=steps,
        batch_size=batch_size,
        context_size=context_size
    )

    # Convert the prompt into tiktoken IDs
    prompt_ids = encode(prompt)

    # Shape: (1, number_of_prompt_tokens)
    start_idx = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    generated_ids = generate(
        model=model,
        context_size=context_size,
        start_idx=start_idx,
        number_of_tokens=max_new_tokens
    )

    generated_text = decode(
        generated_ids[0].tolist()
    )

    print(generated_text)

In [ ]:
#for special tokens

def train_generate_print(
    model,
    ingredients,
    steps=3000,
    batch_size=4,
    context_size=128,
    max_new_tokens=300
):
    train(
        model=model,
        steps=steps,
        batch_size=batch_size,
        context_size=context_size
    )

    if isinstance(ingredients, str):
        ingredient_list = [
            ingredient.strip()
            for ingredient in ingredients.split(",")
            if ingredient.strip()
        ]
    else:
        ingredient_list = ingredients

    ingredient_text = "\n".join(
        ingredient_list
    )

    prompt = (
        "<|recipe_start|>\n"
        "<|ingredients|>\n"
        f"{ingredient_text}\n"
        "<|title|>\n"
    )

    prompt_ids = encode(prompt)

    start_idx = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    end_token_id = SPECIAL_TOKENS[
        "<|recipe_end|>"
    ]

    generated_ids = generate(
        model=model,
        context_size=context_size,
        start_idx=start_idx,
        number_of_tokens=max_new_tokens,
        end_token_id=end_token_id
    )

    generated_text = decode(
        generated_ids[0].tolist()
    )

    print(generated_text)

In [ ]:
train_generate_print(
    model=model,
    ingredients=[
        "chicken breast",
        "rice",
        "carrots",
        "garlic"
    ],
    steps=5000,
    batch_size=8,
    context_size=128,
    max_new_tokens=300
)

In [53]:
m = BigramLanguageModel(vocab_size).to(device)
train_generate_print(m)

RuntimeError: Invalid buffer size: 9.41 GiB

Self Attention Mechanism

In [104]:
# self-attention for single individual head
torch.manual_seed(1234)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
# B batch, T sequence length, C n_embd

**query** - what information am i looking for
e.g. in character world that could be 

**key** - what do i have

**value** - what information do i want to propagate


In [105]:
head_size = 16
key = nn.Linear(C, head_size, bias=False) # learnable linear transformation ~ learnable function
query = nn.Linear(C, head_size, bias=False)

k = key(x) # (B,T,16)
q = query(x) # (B,T,16)

wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) = (B, T, T)

In [106]:
wei[0]

tensor([[ 1.0937,  0.5420,  0.5606, -0.9620,  1.9142, -0.3077,  0.3238,  0.8186],
        [-0.6120,  0.3489,  1.4232, -0.6161,  3.3533,  0.1684,  0.2174, -1.2803],
        [ 1.7824,  3.0182,  0.7259, -1.3615,  1.0162, -0.9429,  0.9141,  1.4707],
        [-0.6378, -2.6615,  0.6544, -0.2874, -1.6264, -1.3580,  0.1080, -0.8867],
        [-1.2721, -0.7376, -0.0143, -0.6883,  1.0224,  0.0627,  0.1387, -0.1410],
        [ 0.0425,  1.9774, -0.3077, -1.1094, -0.0862, -0.0966,  0.5417,  0.5010],
        [-1.2123, -1.3356, -0.2712,  1.3814,  0.5958, -1.0527, -0.4535, -0.1302],
        [ 0.9748,  1.1991, -0.9978,  0.4849, -1.4779, -0.0939, -0.7011,  0.5800]],
       grad_fn=<SelectBackward0>)

In [107]:
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

In [108]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2767, 0.7233, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.7186, 0.0726, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1615, 0.0213, 0.5879, 0.2292, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0558, 0.0951, 0.1961, 0.1000, 0.5530, 0.0000, 0.0000, 0.0000],
        [0.0935, 0.6474, 0.0659, 0.0296, 0.0822, 0.0814, 0.0000, 0.0000],
        [0.0367, 0.0325, 0.0941, 0.4913, 0.2240, 0.0431, 0.0784, 0.0000],
        [0.2329, 0.2915, 0.0324, 0.1427, 0.0200, 0.0800, 0.0436, 0.1569]],
       grad_fn=<SelectBackward0>)

In [109]:
out = wei @ x # (B, T, T) @ (B, T, C) = (B, T, C)

In [110]:
value = nn.Linear(C, head_size, bias=False)
v = value(x) # (B,T,16)

In [111]:
out = wei @ v # (B, T, T) @ (B, T, C) = (B, T, C)

In [112]:
out

tensor([[[-2.7997e-01, -1.0802e-02, -3.4791e-02, -1.8084e-01, -2.7143e-01,
          -4.2373e-01, -5.3097e-01, -6.8897e-01,  9.6862e-02, -3.4939e-01,
          -6.0128e-02, -1.8804e-01,  4.4252e-01, -9.2242e-01,  3.6726e-01,
           3.3111e-01],
         [-2.2211e-01, -8.5506e-01,  3.7350e-01,  2.5800e-01, -1.3268e-01,
          -1.3579e-01,  5.1694e-01,  2.6489e-01,  6.1621e-01, -9.5768e-01,
          -4.2303e-01, -5.8741e-01, -3.3682e-02, -6.6581e-01,  3.6389e-01,
          -2.5921e-02],
         [-2.2999e-01, -8.8656e-01,  3.4489e-01,  2.9240e-01, -1.2463e-01,
          -1.0228e-01,  5.3278e-01,  3.0596e-01,  6.7497e-01, -9.5598e-01,
          -4.2596e-01, -5.6532e-01, -6.2738e-02, -6.5353e-01,  4.2743e-01,
          -2.9587e-02],
         [-2.6014e-01, -3.3440e-01, -1.1374e-01,  3.4111e-01, -8.0194e-02,
          -1.4641e-02, -1.3138e-01, -3.0924e-02,  6.3533e-01, -3.5081e-01,
          -1.5158e-01,  2.6707e-01,  3.5692e-02, -4.3935e-01,  7.8183e-01,
           1.7591e-01],
    

In [113]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # scaling factor - this is "scaled" dot product attention

In [114]:
k.var()

tensor(0.9988)

In [115]:
q.var()

tensor(1.0134)

In [116]:
wei.var()

tensor(0.9901)

In [117]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [118]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [69]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size, n_embd, context_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

In [119]:
#special token

class Head(nn.Module):
    def __init__(
        self,
        head_size,
        n_embd,
        context_size,
        dropout=0.1
    ):
        super().__init__()

        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    context_size,
                    context_size
                )
            )
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        head_size = k.shape[-1]

        weights = (
            q @ k.transpose(-2, -1)
        ) * head_size ** -0.5

        weights = weights.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        weights = F.softmax(
            weights,
            dim=-1
        )

        weights = self.dropout(weights)

        v = self.value(x)

        return weights @ v

In [70]:

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size, n_embd, context_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, context_size) for _ in range(num_heads)])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return out

In [120]:
#speical

class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        num_heads,
        head_size,
        n_embd,
        context_size,
        dropout=0.1
    ):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(
                head_size,
                n_embd,
                context_size,
                dropout
            )
            for _ in range(num_heads)
        ])

        self.projection = nn.Linear(
            n_embd,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        return self.dropout(
            self.projection(x)
        )

In [71]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

In [121]:
#speical

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        dropout=0.1
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [72]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head, context_size):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, context_size)
        self.ffwd = FeedFoward(n_embd)

    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ffwd(x)
        return x

In [122]:
#special
class Block(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        context_size,
        dropout=0.1
    ):
        super().__init__()

        if n_embd % n_head != 0:
            raise ValueError(
                "n_embd must be divisible by n_head"
            )

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size,
            n_embd,
            context_size,
            dropout
        )

        self.ffwd = FeedForward(
            n_embd,
            dropout
        )

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x

In [73]:
class GPT(nn.Module):
  def __init__(self, vocab_size, n_embd=32, context_size=8, n_head=4, n_layer=4):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # lookup table, vocab_size x vocab_size
    self.position_embedding_table = nn.Embedding(context_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head, context_size=context_size) for _ in range(n_layer)])

    self.ln_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape
    # idx (batch_size, context_size)
    tok_emb = self.token_embedding_table(idx) # (batch_size, context_size, vocab_size)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (block_size, n_embd) # 3

    x = tok_emb + pos_emb

    x = self.blocks(x)

    logits = self.ln_head(x)

    if targets is not None:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    else:
      loss = None

    return logits, loss

In [128]:
#special
class GPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd=128,
        context_size=256,
        n_head=4,
        n_layer=4,
        dropout=0.1
    ):
        super().__init__()

        self.context_size = context_size

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.position_embedding_table = nn.Embedding(
            context_size,
            n_embd
        )

        self.blocks = nn.Sequential(*[
            Block(
                n_embd=n_embd,
                n_head=n_head,
                context_size=context_size,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape

        if T > self.context_size:
            raise ValueError(
                f"Input length {T} exceeds context "
                f"size {self.context_size}."
            )

        token_embeddings = (
            self.token_embedding_table(idx)
        )

        position_ids = torch.arange(
            T,
            device=idx.device
        )

        position_embeddings = (
            self.position_embedding_table(
                position_ids
            )
        )

        x = (
            token_embeddings
            + position_embeddings
        )

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

In [126]:
@torch.no_grad()
def generate_recipe_tokens(
    model,
    prompt_ids,
    max_new_tokens=300,
    temperature=0.8,
    top_k=50
):
    model.eval()

    idx = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    end_token_id = SPECIAL_TOKENS[
        "<|recipe_end|>"
    ]

    for _ in range(max_new_tokens):
        idx_cond = idx[
            :,
            -model.context_size:
        ]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        logits = logits / temperature

        if top_k is not None:
            values, _ = torch.topk(
                logits,
                min(top_k, logits.shape[-1])
            )

            cutoff = values[:, [-1]]

            logits = logits.masked_fill(
                logits < cutoff,
                float("-inf")
            )

        probabilities = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        idx = torch.cat(
            [idx, next_token],
            dim=1
        )

        if next_token.item() == end_token_id:
            break

    model.train()

    return idx[0].tolist()

In [124]:
#special token
#ingredient input function

def generate_recipe(
    model,
    ingredients,
    max_new_tokens=300,
    temperature=0.8,
    top_k=50
):
    if isinstance(ingredients, str):
        ingredient_lines = [
            item.strip()
            for item in ingredients.split(",")
            if item.strip()
        ]
    else:
        ingredient_lines = [
            str(item).strip()
            for item in ingredients
            if str(item).strip()
        ]

    ingredient_text = "\n".join(
        ingredient_lines
    )

    prompt = (
        "<|recipe_start|>\n"
        "<|ingredients|>\n"
        f"{ingredient_text}\n"
        "<|title|>\n"
    )

    prompt_ids = encode(prompt)

    generated_ids = generate_recipe_tokens(
        model=model,
        prompt_ids=prompt_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k
    )

    full_text = decode(generated_ids)

    return full_text

In [129]:
recipe = generate_recipe(
    model=m,
    ingredients=[
        "chicken breast",
        "rice",
        "carrots",
        "garlic"
    ],
    max_new_tokens=300,
    temperature=0.8,
    top_k=50
)

print(recipe)

AttributeError: 'GPT' object has no attribute 'context_size'

In [35]:
#character level
m = GPT(vocab_size).to(device)
train_generate_print(m)

Step 0, train loss: 5.8217 val loss: 5.7929
Step 1000, train loss: 2.1396 val loss: 2.1643
Step 2000, train loss: 1.9435 val loss: 1.9582
Step 3000, train loss: 1.8404 val loss: 1.8233
Step 4000, train loss: 1.7459 val loss: 1.7657
Step 4999, train loss: 1.7057 val loss: 1.7210

4 teaspoons bin broth tog peppers pinilllo for with water until to pockper
2 teaspoon about 10 minuteshlove puy, about 6000 minutes. Ditato four over pan; s howll saucepas froff 1 ther saly. Cooon tire of asplapowservegel, garooil coolaspure.

Title: Smixti and gary of en heaskigh
1 teaspoons dough 


In [74]:
#tiktoken
context_size = 128

m = GPT(
    vocab_size=vocab_size,
    n_embd=128,
    context_size=context_size,
    n_head=4,
    n_layer=4
).to(device)

train_generate_print(
    model=m,
    steps=5000,
    batch_size=16,
    context_size=context_size,
    max_new_tokens=200,
    prompt="<RECIPE>\nTitle:"
)

Step 0, train loss: 11.2674 val loss: 11.2594
Step 1000, train loss: 2.6577 val loss: 2.8014
Step 2000, train loss: 2.3908 val loss: 2.4433
Step 3000, train loss: 2.2348 val loss: 2.2968
Step 4000, train loss: 2.1544 val loss: 2.2149
Step 4999, train loss: 2.0554 val loss: 2.1777
<RECIPE>
Title: Crispy corner Butter Cock Redy

Ingredients: 1 cup all-purpose flour
1/4 teaspoon baking powder (such as Vidalia)
1 cup all-purpose flour
1/2 teaspoon baking soda
1 teaspoon baking powder
1 teaspoon baking soda
1/4 teaspoon ground cumin
1/4 teaspoon salt
1 teaspoon cinnamon
1/4 cup butter, unwuff as Pisting
1 1/2 cups crispy salt
1 cup white sugar
3/4 cup butter, softened

Directions: Sift together flour, almond meal, baking soda, coconut oil, and salt. Toss in a small saucepan over medium-high heat until melted and creamy. Sprinkle over the raisins, and knead until smooth. Toss in all the ingredients to 1 tablespoon of the proper evenly, egg.
Add sugar, and eggs in a saucepan. Whisk and tomato

In [130]:
context_size = 256

m = GPT(
    vocab_size=vocab_size,
    n_embd=128,
    context_size=context_size,
    n_head=4,
    n_layer=4,
).to(device)

In [136]:
train_generate_print(
    model=m,
    ingredients=[
        "chicken breast",
        "rice",
        "carrots",
        "garlic"
    ],
    steps=5000,
    batch_size=4,
    context_size=context_size,
    max_new_tokens=300
)

Step 0, train loss: 2.4180 val loss: 2.4115
Step 1000, train loss: 2.3383 val loss: 2.3199
Step 2000, train loss: 2.2411 val loss: 2.2359
Step 3000, train loss: 2.1423 val loss: 2.2115
Step 4000, train loss: 2.1664 val loss: 2.1227
Step 4999, train loss: 2.0173 val loss: 2.0603
<|recipe_start|>
<|ingredients|>
chicken breast
rice
carrots
garlic
<|title|>
Mels
<|instructions|>
Gather the ingredients. Preheat the oven to 375 degrees F (190 degrees C).
Heat milk, add blackberries, lemon zest, tea bags, Worcestershire sauce, paprika, black pepper, salt, ground black pepper, salt, and freshly ground coriander. Stir in masa harina and chicken broth.
Cover and cook until cabbage has tender and turn meat hits seeds are tender, 5 to 10 minutes. Peel into each apple.
Mix roasted vegetables, pear, and pepper together in a large bowl. Stir in chicken and Parmesan cheese. Spoon season with salt and pepper in another food glass.
Melt a large pot over medium heat. Add chicken stockpot over medium-hig

In [ ]:
recipe = generate_recipe(
    model=m,
    ingredients=[
        "chicken breast",
        "rice",
        "carrots",
        "garlic"
    ],
    max_new_tokens=300,
    temperature=0.8,
    top_k=50
)

print(recipe)

In [137]:
recipe = generate_recipe(
    model=m,
    ingredients=[
        "salmon",
        "lemon",
        "garlic",
        "potatoes"
    ],
    max_new_tokens=300,
    temperature=0.8,
    top_k=50
)

print(recipe)

<|recipe_start|>
<|ingredients|>
salmon
lemon
garlic
potatoes
<|title|>
Protein Chicken Salad
<|instructions|>
Gather all ingredients.
Place strawberries, oranges, and lemon zest in a large bowl.
Pour mixture into prepared pan and let rest for 1 hour.
Make the sauce: Pour the peach mixture into a saucepan over high heat and bring to a boil. Cover and simmer for 15 minutes.
Add carrots, bell pepper, and celery. Cook, undrained; bring to a boil, stirring constantly until tender, about 10 minutes. Reduce heat to medium-low; simmer until beans are tender.
Stir cream into the skillet; add chicken stock. Serve with Cheddar wedges if desired.
<|recipe_end|>
